<a href="https://www.kaggle.com/code/pavankumar960/happiness-index-prediction-in-south-asia-middle?scriptVersionId=240121681" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [ ]:
# installing missing libraries
!pip install gradio

# Importing Libraries

In [ ]:
import pandas as pd
import numpy as np
import gradio as gr

import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression

import warnings
warnings.filterwarnings("ignore")

sns.set(style='whitegrid')
plt.rcParams["figure.figsize"] = (10, 6)

# Loading Dataset

In [ ]:
df = pd.read_csv("/kaggle/input/sa-me-happiness-index/2023_SouthAsia_MiddleEast_Happiness_Education_Income.csv")

In [ ]:
df.columns = df.columns.str.strip()


# Data Preprocessing

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.head()

**Unique Values**

In [ ]:
for column in df.columns:
    print("\n")
    print(df[column].unique())
    print("\n","_"*80)

# Data Cleaning

In [ ]:
df.isnull().sum()

In [ ]:
missing_rows = df[df.isnull().any(axis=1)]
print(missing_rows)

In [ ]:
df1 = df.dropna()

In [ ]:
df.duplicated().sum()

# Data Visualization

## Univariate Analysis

In [ ]:
df['Region'].value_counts().plot(kind='bar', title='Region Distribution')
plt.xlabel('Region')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
numerical_cols = [
    'Happiness_Score', 'GDP_per_Capita_USD', 'Social_Support',
    'Literacy_Rate(%)', 'Offline_School_Enrollment(%)',
    'Online_Education_Access(%)', 'Median_Income_USD',
    'Unemployment_Rate(%)', 'Poverty_Rate(%)'
]

for col in numerical_cols:
    sns.histplot(df[col].dropna(), kde=True)
    plt.title(f'Distribution of {col}')
    plt.xlabel(col)
    plt.ylabel('Frequency')
    plt.tight_layout()
    plt.show()


## Bivariate Analysis

In [ ]:
features_to_compare = [
    'GDP_per_Capita_USD', 'Social_Support', 'Literacy_Rate(%)',
    'Offline_School_Enrollment(%)', 'Online_Education_Access(%)',
    'Median_Income_USD', 'Unemployment_Rate(%)', 'Poverty_Rate(%)'
]

for col in features_to_compare:
    sns.scatterplot(data=df, x=col, y='Happiness_Score')
    plt.title(f'Happiness Score vs {col}')
    plt.xlabel(col)
    plt.ylabel('Happiness Score')
    plt.tight_layout()
    plt.show()

In [ ]:
sns.boxplot(data=df, x='Region', y='Happiness_Score')
plt.xticks(rotation=0)
plt.title('Happiness Score by Region')
plt.tight_layout()
plt.show()

## Multivariate Analysis

In [ ]:
corr = df[numerical_cols].corr()

plt.figure(figsize=(12, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap='rocket', linewidths=0.5)
plt.title('Correlation Heatmap')
plt.tight_layout()
plt.show()


# Model Training

In [ ]:
features = ['GDP_per_Capita_USD', 'Social_Support', 'Literacy_Rate(%)', 
            'Offline_School_Enrollment(%)', 'Online_Education_Access(%)',
            'Median_Income_USD', 'Unemployment_Rate(%)', 'Poverty_Rate(%)']
target = 'Happiness_Score'

X = df1[features]
y = df1[target]

model = LinearRegression()
model.fit(X, y)

In [ ]:
for f, c in zip(features, model.coef_):
    print(f"{f}: {c:.4f}")

print("Intercept:", model.intercept_)


# Model Deployment

In [ ]:
def predict_happiness(gdp, support, literacy, offline, online, income, unemp, poverty):
    input_data = [[gdp, support, literacy, offline, online, income, unemp, poverty]]
    prediction = model.predict(input_data)[0]
    return round(prediction, 2)

gr.Interface(
    fn=predict_happiness,
    inputs=[
        gr.Slider(0, 100000, label="GDP per Capita (USD)"),
        gr.Slider(0, 1, step=0.01, label="Social Support"),
        gr.Slider(0, 100, label="Literacy Rate (%)"),
        gr.Slider(0, 100, label="Offline School Enrollment (%)"),
        gr.Slider(0, 100, label="Online Education Access (%)"),
        gr.Slider(0, 100000, label="Median Income (USD)"),
        gr.Slider(0, 100, label="Unemployment Rate (%)"),
        gr.Slider(0, 100, label="Poverty Rate (%)"),
    ],
    outputs="number",
    title="Happiness Score Predictor",
    description="Note: This model is trained on a small demo dataset of 14 countries. Results are not accurate, but for educational use."
).launch(share=True)  # Use `share=False` in Kaggle if errors occur


# Conclusion

- The dataset consists of only 14 countries, which is too small for reliable model training or generalization.

- Two countries had missing values in key features like GDP per Capita and Median Income, and were excluded from the analysis.

- The target variable is Happiness Score.

- Features like Unemployment Rate and Poverty Rate show a negative correlation with Happiness, which aligns with real-world expectations higher unemployment and poverty tend to reduce happiness.